In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

In [2]:
# Make project imports work both locally and in Colab.
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "nepse_analyst").exists() is False:
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from nepse_analyst.database import create_database
from nepse_analyst.config import DB_PATH

In [3]:
# 1) Initialize schema.
create_database()

# 2) Resolve CSV path.
csv_path = PROJECT_ROOT / "data" / "raw" / "nepse_companies_kaggle.csv"
if not csv_path.exists():
    # Colab fallback: upload manually or mount Google Drive and set this path.
    colab_candidates = [
        Path("/content/nepse_companies_kaggle.csv"),
        Path("/content/drive/MyDrive/nepse_companies_kaggle.csv"),
    ]
    existing = [p for p in colab_candidates if p.exists()]
    if not existing:
        raise FileNotFoundError(
            "CSV not found. Place nepse_companies_kaggle.csv in data/raw/ or /content/."
        )
    csv_path = existing[0]

# 3) Read and inspect source columns.
df = pd.read_csv(csv_path)
print("Raw columns:", list(df.columns))
print("Raw rows:", len(df))

# 4) Normalize source columns and map to schema names.
normalized_columns = {c: c.strip().lower().replace(" ", "_") for c in df.columns}
df = df.rename(columns=normalized_columns)

Raw columns: ['name', 'company_nmae', 'symbol', 'Listed_Shares', 'Paidup_Value', 'Total_Paidup']
Raw rows: 520


In [4]:
# Kaggle file has typo company_nmae and uses "name" for sector.
column_mapping = {
    "name": "sector",
    "company_nmae": "name",
    "symbol": "symbol",
    "listed_shares": "listed_shares",
    "paidup_value": "paid_up_value",
    "total_paidup": "total_paid_up",
}

In [5]:
missing = [c for c in column_mapping if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected CSV columns: {missing}")

df = df[list(column_mapping.keys())].rename(columns=column_mapping)

In [6]:
# 5) Clean and standardize values.
for col in ["symbol", "name", "sector"]:
    df[col] = df[col].astype(str).str.strip()

df["symbol"] = df["symbol"].str.upper()

df["listed_shares"] = (
    pd.to_numeric(df["listed_shares"], errors="coerce").fillna(0).astype(int)
)
df["paid_up_value"] = pd.to_numeric(df["paid_up_value"], errors="coerce")
df["total_paid_up"] = pd.to_numeric(df["total_paid_up"], errors="coerce")

In [7]:
# Map Kaggle sector labels to PRD sector names.
sector_map = {
    "Commercial Banks": "Commercial Banks",
    "Hydro Power": "Hydropower",
    "Life Insurance": "Life Insurance",
    "Development Bank Limited": "Development Banks",
    "Finance": "Finance Companies",
    "Microfinance": "Microfinance",
    "Non-Life Insurance": "Non-Life Insurance",
    "Hotels And Tourism": "Hotels & Tourism",
    "Manufacturing And Processing": "Manufacturing",
    "Tradings": "Trading",
    "Mutual Fund": "Mutual Funds",
    "Investment": "Investment",
    "Others": "Others",
    "Corporate Debenture": "Corp. Debentures",
    "Government Bond": "Govt. Bonds",
    "Capital": "Others",
    "Promotor Share": "Others",
}

df["sector"] = df["sector"].replace(sector_map)
df["is_active"] = 1

In [8]:
# Keep one row per symbol to avoid PK conflicts.
df = df.drop_duplicates(subset=["symbol"], keep="first")

In [9]:
# 6) Insert into SQLite.
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("DELETE FROM companies")
conn.commit()

df.to_sql("companies", conn, if_exists="append", index=False)

520

In [10]:
# 7) Verify load and sector normalization.
count_df = pd.read_sql_query("SELECT COUNT(*) AS companies_count FROM companies", conn)
sectors_df = pd.read_sql_query(
    "SELECT DISTINCT sector FROM companies ORDER BY sector",
    conn,
)

In [11]:
print("\nCompanies inserted:")
print(count_df)
print("\nDistinct sectors:")
print(sectors_df)

conn.close()


Companies inserted:
   companies_count
0              520

Distinct sectors:
                sector
0     Commercial Banks
1     Corp. Debentures
2    Development Banks
3    Finance Companies
4          Govt. Bonds
5     Hotels & Tourism
6           Hydropower
7           Investment
8       Life Insurance
9        Manufacturing
10        Microfinance
11        Mutual Funds
12  Non-Life Insurance
13              Others
14             Trading
